# OperatorLab: Zero-Shot Resolution Generalization Demo

This demo shows the core breakthrough of Neural Operators:
**Learning mappings between infinite-dimensional function spaces**.

Unlike standard CNNs that fail when evaluated on unseen grid dimensions, a Neural Operator trained at low resolution ($16\times 16$ or $32\times 32$) can be evaluated **zero-shot** at higher resolutions ($64\times 64$, $128\times 128$) without retraining.

In [ ]:
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

from operatorlab.models.fno import FNO2d
from operatorlab.physics.heat import HeatEquation2D
from operatorlab.data.datasets import InMemoryDataset
from operatorlab.training.losses import OperatorLoss, relative_l2
from operatorlab.training.trainer import Trainer
from operatorlab.evaluation.resolution_sweep import resolution_sweep

## 1. Problem Setup: 2D Heat Diffusion Equation

$$\partial_t u = \alpha \nabla^2 u, \quad u(x, y, 0) = a(x, y)$$

We generate synthetic training data at $16\times 16$ resolution.

In [ ]:
pde = HeatEquation2D(alpha=0.01, T=0.5)
train_data = pde.generate_dataset(n_samples=100, resolution=16, seed=42)
val_data = pde.generate_dataset(n_samples=25, resolution=16, seed=43)

train_ds = InMemoryDataset(train_data["a"], train_data["u"], normalize=True)
val_ds = InMemoryDataset(val_data["a"], val_data["u"], normalize=True)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16)

## 2. Train Fourier Neural Operator (FNO)

We train a compact 4-layer FNO model.

In [ ]:
model = FNO2d(modes1=4, modes2=4, width=32, n_layers=4)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = OperatorLoss(l2_weight=1.0, h1_weight=0.01)

trainer = Trainer(model=model, optimizer=optimizer, loss_fn=loss_fn, device="cpu", use_amp=False)
result = trainer.train(train_loader, val_loader, n_epochs=10)
print(f"Final train loss: {result.final_train_loss:.6f}, Best val loss: {result.best_val_loss:.6f}")

## 3. Zero-Shot Super-Resolution Evaluation

Evaluate the trained model directly on higher grid resolutions ($16\times 16$, $32\times 32$, $64\times 64$) without retraining.

In [ ]:
sweep = resolution_sweep(
    model=model,
    pde=pde,
    base_resolution=16,
    target_resolutions=[16, 32, 64],
    n_test_samples=20,
    device="cpu",
)
print(sweep.summary_table())

## 4. Visualizing Resolution Invariance

Plotting the relative $L_2$ error as a function of target resolution.

In [ ]:
resolutions = [r.resolution for r in sweep.results]
errors = [r.l2_error for r in sweep.results]

plt.figure(figsize=(6, 4))
plt.plot(resolutions, errors, marker="o", color="royalblue", linewidth=2)
plt.title("Zero-Shot Resolution Scaling (Trained on 16x16)")
plt.xlabel("Resolution")
plt.ylabel("Relative L2 Error")
plt.grid(True, alpha=0.3)
plt.show()